# AF2LUM — isolated Kaggle arm

In [ ]:
import os,shutil,subprocess,sys,time
from pathlib import Path
ARM='AF2LUM'; WORK=Path('/kaggle/working'); REPO=WORK/'coffee-bean-detection'; INPUT=Path('/kaggle/input'); OUT=WORK/'af2-spectral-factorization-v1'
if REPO.exists(): shutil.rmtree(REPO)
for _ in range(3):
 if subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]).returncode==0: break
 if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True); os.chdir(REPO)
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input,restore_spectral_kaggle_run
from coffee_detector.af2_spectral.audit import run_spectral_static_audit
DATA,A,_=prepare_af2_spectral_kaggle_input(INPUT,WORK); OUT.mkdir(exist_ok=True); STATIC=OUT/'static_audit.json'; assert run_spectral_static_audit(A['D0_seed42_best.pt'],STATIC,device='cuda:0')['decision']=='PASS'
CONFIG=REPO/f'configs/af2_spectral/{ARM}_yolo26n.yaml'; restore_spectral_kaggle_run(INPUT,OUT,arm=ARM,seed=42,d0_checkpoint=A['D0_seed42_best.pt'],config=CONFIG)
LOG=OUT/f'{ARM}_seed42_run.log'; RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'; CMD=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spectral_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(A['D0_seed42_best.pt']),'--static-audit',str(STATIC),'--output-root',str(OUT),'--device','0','--authorize-training']
if not RESULT.is_file():
 with LOG.open('a') as s: p=subprocess.Popen(CMD,cwd=REPO,stdout=s,stderr=subprocess.STDOUT)
 seen=-1
 while p.poll() is None:
  q=OUT/ARM/f'{ARM}_seed42'/'results.csv'; n=max(0,len(q.read_text().splitlines())-1) if q.is_file() else 0
  if n!=seen: print(f'{ARM}: {n}/50 epoch | log={LOG}',flush=True); seen=n
  time.sleep(120)
 if p.returncode: print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal')
assert RESULT.is_file(); print(RESULT.read_text()); print('DOWNLOAD SEBELUM STOP SESSION:',shutil.make_archive(f'/kaggle/working/{ARM}_seed42_output','zip',OUT))